In [0]:
%pip install                                                         \
        numpy                 sentencepiece       protobuf           \
        adlfs                 fsspec              azure-storage-blob \
        readability-lxml      w3lib               httpx              \
        beautifulsoup4        azure-ai-inference  packaging          \
        python-dotenv         openai              polars             \
        sentence-transformers tiktoken            fastembed

%restart_python

In [0]:
import os
import gc
import ast
import json
import asyncio
import shutil
import httpx
import numpy  as np
import scipy  as sp
from dotenv                    import load_dotenv
from pathlib                   import Path
from openai                    import AsyncOpenAI

import polars as pl
import matplotlib.pyplot as plt

import pandas as pd
from pyspark.sql               import SparkSession, Window
from pyspark.sql.functions     import col
import pyspark.sql.functions as F
from delta                     import configure_spark_with_delta_pip

from azure.ai.inference.models import SystemMessage, UserMessage
from sentence_transformers     import SentenceTransformer

from llm_agent                 import LlmAgent

In [ ]:
builder = SparkSession.builder\
            .config("spark.sql.sources.commitProtocolClass", "org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol")\
            .config("spark.sql.parquet.output.committer.class", "org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter")\
            .config("spark.mapreduce.fileoutputcommitter.marksuccessfuljobs","false")\
            .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")\
            .config("spark.sql.adaptive.enabled", True)\
            .config("spark.sql.shuffle.partitions", "auto")\
            .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "100MB")\
            .config("spark.sql.adaptive.coalescePartitions.enabled", True)\
            .config("spark.sql.dynamicPartitionPruning.enabled", True)\
            .config("spark.sql.autoBroadcastJoinThreshold", "10MB")\
            .config("spark.sql.session.timeZone", "Asia/Tokyo")\
            .config("spark.sql.execution.arrow.pyspark.enabled", "true")\
            .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")\
            .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")\
            .config("spark.databricks.delta.write.isolationLevel", "SnapshotIsolation")\
            .config("spark.databricks.delta.optimizeWrite.enabled", True)\
            .config("spark.databricks.delta.autoCompact.enabled", True)
            # Delta Lake 用の SQL コミットプロトコルを指定
            # Parquet 出力時のコミッタークラスを指定
            # Azure Blob Storage (ABFS) 用のコミッターファクトリを指定
            # '_SUCCESS'で始まるファイルを書き込まないように設定
            # JavaSerializerからKryoSerializerへの変更
            # AQE(Adaptive Query Execution)の有効化
            # パーティション数を自動で調整するように設定
            # シャッフル後の1パーティションあたりの最小サイズを指定
            # AQEのパーティション合成の有効化
            # 動的パーティションプルーニングの有効化
            # 小さいテーブルのブロードキャスト結合への自動変換をするための閾値調整
            # SparkSessionのタイムゾーンを日本標準時刻に設定
            # Apache Arrow 形式で Pandas<=>Spark 変換を行う
            # Delta Lake固有のSQL構文や解析機能を拡張モジュールとして有効化
            # SparkカタログをDeltaLakeカタログへ変更
            # Delta Lake書き込み時のアイソレーションレベルを「スナップショット分離」に設定
            # 書き込み時にデータシャッフルを行い、大きなファイルを生成する機能の有効化
            # 書き込み後に小さなファイルを自動で統合する機能の有効化

# 追加の設定処理
# クラスターサイズによって、動的に書き換えること
builder = builder\
			.config("spark.driver.memory", "64g")\
    		.config("spark.driver.maxResultSize", "32g")\
			.config("spark.rpc.message.maxSize", "1024")

spark = configure_spark_with_delta_pip(builder).getOrCreate()


In [0]:
# .env ファイルを読み込む
load_dotenv()

# 環境変数の取得
AI_FOUNDRY_ENDPOINT       = os.environ.get("AI_FOUNDRY_ENDPOINT")
AI_FOUNDRY_API_KEY        = os.environ.get("AI_FOUNDRY_API_KEY")
AI_FOUNDRY_MODEL          = os.environ.get("AI_FOUNDRY_MODEL")
LLM_MAX_TOKENS            = os.environ.get("LLM_MAX_TOKENS")
LLM_TEMPERATURE           = os.environ.get("LLM_TEMPERATURE")
LLM_TOP_P                 = os.environ.get("LLM_TOP_P")

# ウェジットからの設定の読み込み
BRAND_NAME                = dbutils.widgets.get('BRAND_NAME')
TARGET_BRAND_WORDS        = dbutils.widgets.get('TARGET_BRAND_WORDS')
EXTRACTION_ID             = dbutils.widgets.get('EXTRACTION_ID')

# メモ：
# 環境変数・ウィジット経由でのパラメータの取得方法では、無条件にパラメータの型がStringに変換されてしまう
# そのため、受け取ったパラメータを適切に型変換する必要がある
LLM_MAX_TOKENS            = int(LLM_MAX_TOKENS)
LLM_TEMPERATURE           = float(LLM_TEMPERATURE)
LLM_TOP_P                 = float(LLM_TOP_P)
EXTRACTION_ID             = int(EXTRACTION_ID)

# メモ：
# TARGET_BRAND_WORDS の中身として
# "['aaa', 'bbb', 'ccc']"
# "aaa" or "'aaaaa'"
# 上記のような「文字列」を想定している
# 
# このような文字列を正しくパースするために、astライブラリを利用することとした
try:
	TARGET_BRAND_WORDS = ast.literal_eval(TARGET_BRAND_WORDS)
	if isinstance(TARGET_BRAND_WORDS, str):
		TARGET_BRAND_WORDS = [TARGET_BRAND_WORDS]
except (ValueError, SyntaxError):
	TARGET_BRAND_WORDS = [TARGET_BRAND_WORDS]

# 簡易デバッグ用
# print(f'AI_FOUNDRY_ENDPOINT:       {AI_FOUNDRY_ENDPOINT}')       # セキュリティリスクのため、コメントアウト
# print(f'AI_FOUNDRY_API_KEY:        {AI_FOUNDRY_API_KEY}')        # セキュリティリスクのため、コメントアウト
print(f'AI_FOUNDRY_MODEL:          {AI_FOUNDRY_MODEL}')
print(f'LLM_MAX_TOKENS:            {LLM_MAX_TOKENS}')
print(f'LLM_TEMPERATURE:           {LLM_TEMPERATURE}')
print(f'LLM_TOP_P:                 {LLM_TOP_P}')
print(f'BRAND_NAME:                {BRAND_NAME}')
print(f'TARGET_BRAND_WORDS:        {TARGET_BRAND_WORDS}')
print(f'EXTRACTION_ID:             {EXTRACTION_ID}')

In [0]:
limits      = httpx.Limits(max_keepalive_connections=20, max_connections=300)
timeout     = httpx.Timeout(300.0, connect=5.0)
http_client = httpx.AsyncClient(limits=limits, timeout=timeout)
openai_cli  = AsyncOpenAI(
                    base_url=AI_FOUNDRY_ENDPOINT,
                    api_key=AI_FOUNDRY_API_KEY,
                    http_client=http_client,
                    max_retries=3
                )
semaphore   = asyncio.Semaphore(50)
llmClient   = LlmAgent(openai_cli, AI_FOUNDRY_MODEL, int(LLM_MAX_TOKENS), float(LLM_TEMPERATURE), float(LLM_TOP_P), semaphore)



In [ ]:
output_file_path = f'ブランドLP/{BRAND_NAME}.md'

# 商品分析情報が存在しない場合
if not os.path.exists(output_file_path):
	with open(f'ブランドLP/商品分析.md', 'r', encoding='utf-8') as f:
		sys_prompt = f.read()
		sys_prompt = sys_prompt.replace('【WORD_LIST】', ', '.join(TARGET_BRAND_WORDS))

	messages  = []
	messages.append(SystemMessage(content=sys_prompt))
	product_analysis = await llmClient.complete(messages)
	with open(output_file_path, 'w', encoding='utf-8') as f:
		f.write(json.dumps(product_analysis, indent=4, ensure_ascii=False))

# 商品分析情報の読み込み
with open(output_file_path, 'r', encoding='utf-8') as f:
    product_analysis = f.read()
product_analysis

In [0]:
messages  = []
messages.append(SystemMessage(content=(
				"あなたは提供された「商品分析結果」を基に、その商品の利用文脈（Micro-Context）を特定するマーケティングの専門家です。\n"
				"提供された商品情報を分析し、その商品が「最高に輝く具体的なシーン（適合）」と「全く役に立たない、あるいはミスマッチなシーン（不適合）」を洗い出してください。\n\n"
				
				"【重要な指示】\n"
                "出力するキーワードは、ベクトル検索のクエリとして使用されます。\n"
				"そのため、単一の一般名詞（例：「公園」「オフィス」）は **禁止** です。\n"
                "必ず **「場所＋状況」** または **「属性＋場所」** の複合キーワード（Micro-Context）を選定してください。\n"
    			"抽出する単語は、単なる一般名詞ではなく、「誰が・どこで・何をしているか」がありありと想像できるような、具体的かつ解像度の高いキーワードを選定してください。\n"
    			"特に「場所」に関しては、大分類（例：公園）ではなく、詳細な施設タイプ（例：ドッグラン、親水広場）や、利用目的が明確なスポット名を優先してください。\n"
                "LPに直接記載がなくても、商品の特性から論理的に推測されるシーンは積極的に広げて記述してください。\n"
                "・NG例: 「ジム」「キャンプ」「サラリーマン」\n"
                "・OK例: 「深夜の24時間ジム」「雨上がりのオートキャンプ場」「満員電車の通勤客」「コンセントのあるカフェ席」\n\n"

				"【出力形式（厳守）】\n"
				"回答は必ず以下のJSON形式のみを出力してください。\n"
				"各単語をキーとし、その関連度の強さ（重み）を 0.0〜1.0 の数値で値として設定してください。\n"
				"Markdown記法（```json 等）は含めず、生のJSONテキストのみを返してください。\n"
                "・Positiveとして、確信度の高い上位20個程度を抽出してください。\n"
                "・Negativeとして、確信度の高い上位12個程度を抽出してください。\n\n"
					
				"""
				{
					"positive": {"複合キーワードA": 0.89, "キーワードB": 0.70, ...},
					"negative": {"複合キーワードC": 0.91, ...},
				}
				"""
				"\n\n"
				
				"【分析の視点】\n"
				"--- positive（適合）：商品が必須となる、または魅力を最大化する文脈 ---\n"
				"1. 具体的な施設・スポット（Places）：\n"
				"   - 抽象的な「店」「屋外」はNG。\n"
				"   - 「24時間ジム」「オートキャンプ場」「コワーキングスペース」など、行動が特定できる施設名。\n"
				"2. 利用シーン・瞬間（Scenes）：\n"
				"   - 「通勤ラッシュ」「運動後のシャワー」「子供の寝かしつけ」など、具体的なタイムラインや状況。\n"
				"3. ターゲットの属性・状態（Traits）：\n"
				"   - 「健康志向」のような広い言葉より、「糖質制限中」「リモートワーク疲れ」など具体的な状態。\n"
                "4. 物理的適合 (Physical Fit):\n"
                "   - 商品のサイズ、電源、耐久性が、その場所の設備・環境と完璧に噛み合うか。\n"
                "   - マグネットでくっつく防水Bluetoothスピーカー  →  「ユニットバスの壁面」「雨の日のキャンプのタープ下」\n"
				"5. 心理的・行動的適合 (Contextual Fit):\n"
                "   - その場所にいる人の「特定の悩み」を解決するか。\n"
				"   - 周囲の音を消すデジタル耳栓  →  「いびきが気になるカプセルホテル」「瞑想に集中したいヨガスタジオの隅」\n\n"
				
				"--- negative（不適合）：商品の機能が死ぬ、または邪魔になる文脈 ---\n"
				"1. 阻害要因となる場所（Places）：\n"
				"   - 商品のスペック（大きさ、音、電源有無など）的に使えない場所（例：図書館、満員電車）。\n"
				"2. 無意味なシーン（Scenes）：\n"
				"   - その商品をあえて使う必要がない状況。\n"
                "3. 環境不適合 (Environmental Mismatch):\n"
                "   - 商品を使うには狭すぎる、暗すぎる、うるさすぎる、静かすぎる場所。\n"
                "4. マナー・ルール違反 (Social Mismatch):\n"
                "   - その場所でその商品を使うことが「白い目」で見られる、あるいは禁止されている。\n\n"
				
				"【重み（スコア）の基準】\n"
				"・1.0に近いほど：その傾向が非常に強い、確信度が高い\n"
				"・0.0に近いほど：関連性が薄い\n"
                "・1.0: 完全にその商品の独壇場である（または絶対に使用不可である）。\n"
                "・0.8: 非常に相性が良い（または強い懸念がある）。\n"
                "・0.5: 条件による（今回は出力対象外）。\n"
				"・positiveの場合：適合度の高さ\n"
				"・negativeの場合：不適合度の高さ（明確に避けるべき度合い）\n\n"
			)))
messages.append(UserMessage(content=product_analysis))
analysis_data = await llmClient.complete(messages)
analysis_data

In [ ]:
# メモ：
# cohort.npzは以下のようなデータ構成になっている
# cohort.npz
#     |-- data              : 計算済みコホート係数行列
#     |-- indices           : データの位置指定子(列)
#     |-- indptr            : 行ごとのスライスインデックス
#     |-- shape             : コホート係数行列の形(ADIDのリスト × コホートキャプションのリスト)
#     |-- adid_list         : ADIDのリスト(列)
#     |-- business_codelist : コホートキャプションIDのリスト(行)
TARGET      = "/Volumes/stgadintedmpadintedi/featurestore/behaviorvector/cohort.npz"
npz         = np.load(TARGET, allow_pickle=True)
np_cohort   = sp.sparse.csr_matrix((npz["data"], npz["indices"], npz["indptr"]), shape=tuple(npz["shape"]))
np_adidlist = npz["adid_list"]
np_codelist = npz["business_codelist"]
coo_matrix  = np_cohort.tocoo()

sdf_cohort  = spark.createDataFrame(pd.DataFrame({
						"adid"          : np_adidlist[coo_matrix.row],
						"business_code" : np_codelist[coo_matrix.col],
						"value"         : coo_matrix.data
					}))

# メモリ管理
del npz
del np_cohort, np_adidlist, np_codelist
del coo_matrix
gc.collect()

# CODELISTに対応するキャプションの文脈行列を取得
CAPTION_CONTEXT_MATRIX = "cohort_caption_matrix.npz"
npz                    = np.load(CAPTION_CONTEXT_MATRIX, allow_pickle=True)
spots_matrix           = npz["data"]
relational_codelist    = npz["business_codelist"]

N, D                   = spots_matrix.shape
business_codes_long    = np.repeat(relational_codelist, D)
embedding_indices_long = np.tile(np.arange(D), N)
values_long            = spots_matrix.ravel()
sdf_spots_matrix       = spark.createDataFrame(pd.DataFrame({
    							"business_code" : business_codes_long,
								"embedding"     : embedding_indices_long,
								"value"         : values_long
							}))

# 簡易チェック
assert np.array_equal(relational_codelist, np_codelist), "There are unregistered keys."
assert spots_matrix.shape[0] == np_cohort.shape[1], "Dimension mismatch: spots_matrix rows (count: {}) do not align with np_cohort columns (count: {})" .format(spots_matrix.shape[0], np_cohort.shape[1])

# メモリ管理
del npz
del spots_matrix, relational_codelist
del business_codes_long, embedding_indices_long, values_long
gc.collect()


In [ ]:
# 特定の地点に対するADIDリストを取得
sdf_numpylist     = sdf_cohort.select('adid').dropDuplicates()
RELATIONAL_COHORT = "adinte_datainfrastructure.master.relational_cohort"
sdf_observation   = spark.read.table(RELATIONAL_COHORT)\
							.select(['adid', 'count'])\
                            .join(sdf_numpylist, on='adid', how='inner')\
                            .groupBy('adid')\
                            .agg(F.sum(col('count')).alias('count'))\
                            .select(['adid', 'count'])
POLYGONLIST_TABLE = "adinte_adintedmp.envprod.polygonlist"
sdf_polygonlist   = spark.read.table(POLYGONLIST_TABLE)\
							.select(['project', 'caption', 'extraction_id', 'branch_id', 'spot_id', 'lat_top', 'lon_left'])\
                            .filter(col('extraction_id') == EXTRACTION_ID)\
                            .dropDuplicates()

# メモ：
# sourceテーブルはレコードに一定期間の寿命が付与されている
# 予め決められた期間を過ぎるとレコードが削除される仕組みであるようだ
# そのため、できる限りローカルにバックアップデータを保存しておいて、これを参照する形式にすることとした
ADIDLIST_PATH = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Aiico/preprocess/source_table_{EXTRACTION_ID}.parquet"
path_sdf_file = Path(ADIDLIST_PATH).resolve()
if not path_sdf_file.exists():
	ADIDLIST_TABLE = "adinte_adintedmp.envprod.source"
	pdf_adidlist   = spark.read.table(ADIDLIST_TABLE)\
								.filter(col('extraction_id') == EXTRACTION_ID)\
								.toPandas()
	pdf_adidlist.to_parquet(ADIDLIST_PATH, compression="snappy")
	del pdf_adidlist
	gc.collect()

sdf_adidlist = spark.read.parquet(str(path_sdf_file))\
							.select(['extraction_id', 'branch_id', 'spot_id', 'adid'])\
							.filter(col('extraction_id') == EXTRACTION_ID)\
							.join(sdf_observation, on=['adid'], how='inner')\
							.join(sdf_polygonlist, on=['extraction_id', 'branch_id', 'spot_id'], how='inner')\
							.select(['caption', 'adid', 'count', 'lat_top', 'lon_left'])\
							.dropDuplicates()

sdf_adidlist.display()

In [ ]:
# P(ADID)
sdf_adid_obs   = sdf_observation\
							.withColumn('probability', col('count') / F.sum('count').over(Window.partitionBy(F.lit(1))))\
                        	.select(['adid', 'probability'])

# P(ADID  | AIICO)
# P(AIICO | ADID)  ← AIICOには一様分布を仮定している
sdf_adid_and_aiico    = sdf_adidlist\
							.select(['caption', 'adid', 'count'])
sdf_adid_on_aiico_obs = sdf_adid_and_aiico\
							.withColumn('probability', col('count') / F.sum('count').over(Window.partitionBy('adid')))\
                            .select(['caption', 'adid', 'probability'])
sdf_aiico_on_adid_obs = sdf_adid_and_aiico\
							.withColumn('probability', col('count') / F.sum('count').over(Window.partitionBy('caption')))\
                            .select(['caption', 'adid', 'probability'])

sdf_adid_obs

In [0]:
# LLMの出力をベクトル化
items_positive   = list(analysis_data['positive'].items())
items_negative   = list(analysis_data['negative'].items())
lp_keywords      = [key for key, _ in items_positive] + [ key for key, _ in items_negative]
lp_weights       = [val for _, val in items_positive] + [-val for _, val in items_negative]

# L1正則化
total_weight     = np.sum(np.abs(lp_weights))
lp_weights       = np.array(lp_weights) / total_weight

# Embeddingモデルによる高次元空間への写像
model            = SentenceTransformer('BAAI/bge-m3')
lp_vector        = lp_weights.reshape(1, -1)                             # 1 × キーワード数M
lp_matrix        = model.encode(lp_keywords, normalize_embeddings=True)  # キーワード数M × 1024

# LPからみたスポットの重要度ベクトル
lp_coefficient    = lp_vector @ lp_matrix
embedding_indices = np.arange(D)
values_long       = lp_coefficient.ravel()
sdf_lp_coeff      = spark.createDataFrame(pd.DataFrame({
								"embedding" : embedding_indices,
								"value"     : values_long
							}))\
							.withColumn('abs2',     col('value') * col('value'))\
                            .withColumn('sum',      F.sum('abs2').over(Window.partitionBy(F.lit(1))))\
                            .withColumn('sqrt',     F.sqrt('sum'))\
                            .withColumn('lp_coeff', col('value') / col('sqrt'))\
							.select(['embedding', 'lp_coeff'])\
							.join(sdf_spots_matrix, on='embedding', how='inner')\
							.selelct(['business_code', 'embedding', 'value', 'lp_coeff'])\
                            .groupBy(['business_code'])\
                            .agg(F.sum(col('value') * col('lp_coeff')).alias('lp_coeff'))\
                            .selelct(['business_code', 'lp_coeff'])

# メモリ管理
del spots_matrix
del lp_keywords, lp_weights, total_weight
del model, lp_vector, lp_matrix
del lp_coefficient, embedding_indices, values_long
gc.collect()

sdf_lp_coeff.display()

In [ ]:
# ADID毎のスコアを算出
sdf_adid_scored   = sdf_cohort\
						.join(sdf_lp_coeff, on='business_code', how='inner')\
						.select( ['business_code', 'adid', 'lp_coeff', 'value'])\
						.groupBy(['adid'])\
						.agg(F.sum(col('lp_coeff') * col('value')).alias('score'))\
						.select(['adid', 'score'])
sdf_adid_on_spot  = sdf_cohort\
						.select(['adid', 'business_code', 'value'])\
						.withColumn('exp',     F.exp(col('value')))\
						.withColumn('sum',     F.sum(col('exp')).over(Window.partitionBy('business_code')))\
						.withColumn('softmax', col('exp') / col('sum'))\
						.select(['adid', 'business_code', 'softmax'])\
						.join(sdf_observation, on='adid', how='inner')\
						.withColumn('probability', col('softmax') * col('count'))\
						.select(['adid', 'business_code', 'probability'])\
						.withColumn('probability2', col('probability') / F.sum('probability').over(Window.partitionBy('adid')))\
						.select(['adid', 'business_code', 'probability2'])
sdf_aiico_on_spot = sdf_aiico_on_adid_obs\
						.join(sdf_adid_on_spot, on='adid', how='inner')\
                        .select( ['caption', 'business_code', 'adid', 'probability', 'probability2'])\
                        .groupBy(['caption', 'business_code'])\
                        .agg(F.sum(col('probability') * col('probability2')).alias('probability'))\
                        .withColumn('probability', col('probability') / F.sum('probability').over(Window.partitionBy('caption')))
sdf_aiico_on_lp   = sdf_lp_coeff\
						.selelct(['business_code', 'lp_coeff'])\
                        .withColumn('exp',     F.exp(col('lp_coeff')))\
						.withColumn('sum',     F.sum(col('exp')).over(Window.partitionBy('business_code')))\
						.withColumn('softmax', col('exp') / col('sum'))\
                        .join(sdf_aiico_on_spot, on='business_code', how='inner')\
                        .groupBy(['caption'])\
                        .agg(F.sum(col('softmax') * col('probability')).alias('probability'))\
						.withColumn('probability', col('probability') / F.sum('probability').over(Window.partitionBy('caption')))



In [0]:
# 閾値の設定(上位20%地点を指定する)
REASON_THRESHOLD    = np.quantile(sorted_scored, 0.8)

# フレームワークへの変換
sdf_adid_scores     = sdf_adid_scored\
                        .select(['adid', 'score'])\
                        .filter(col('score') > REASON_THRESHOLD)
sdf_caption_scores  = sdf_aiico_on_lp\
                        .join(sdf_adidlist.select(['caption', 'lat_top', 'lon_left']), on='caption', how='inner')\
                        .select(['caption', 'score', 'lat_top', 'lon_left'])\
                        .dropDuplicates()
sdf_adidlist        = sdf_adidlist\
						.join(pldf_adid_scores, on='adid', how='inner')\
                        .select(['caption', 'adid', 'score', 'lat_top', 'lon_left'])\
                        .collect()
# 解析対象地点毎のカウント数を計算
pldf_store_sum      = pldf_adidlist\
						.group_by(pl.col('caption'), maintain_order=True)\
                        .agg(pl.count().alias('total_count'))\
                        .select(['caption', 'total_count'])
count               = pldf_adid_scores.select(pl.col("adid").n_unique()).collect().item()

# メモリ管理
del pldf_location_map
gc.collect()

print(f"処理対象        : {BRAND_NAME}")
print(f"ユニークなADID数 : {count}")
print(pldf_adidlist)

In [ ]:
TARGET = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Aiico/output/{BRAND_NAME}_adid_score.parquet"
pldf_adid_scores.collect().write_parquet(TARGET, compression="snappy")
TARGET = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Aiico/output/{BRAND_NAME}_caption_score.parquet"
pldf_caption_scores.write_parquet( TARGET, compression="snappy")
TARGET = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Aiico/output/{BRAND_NAME}_adid_score_on_caption.parquet"
pldf_adidlist.write_parquet(       TARGET, compression="snappy")
TARGET = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Aiico/output/{BRAND_NAME}_adid_count.parquet"
pldf_store_sum.write_parquet(      TARGET, compression="snappy")